<a href="https://colab.research.google.com/github/MinuraWije/6CS005-HPC/blob/main/Evidence/Notebooks/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 - Password Recovery with CUDA

Recovers passwords encoded by the lecturer's `cudaCrypt` on the GPU, one
thread per password, trying all 26*26*10*10 = 67,600 combinations.
**Runtime:** set Runtime -> Change runtime type -> GPU (T4).

## 0. Repository base URL

`RAW_BASE` points at the public GitHub `Project` folder so the `wget`
cells can download the data and anyone with the link can run this.

In [1]:
RAW_BASE = "https://raw.githubusercontent.com/MinuraWije/6CS005-HPC/main/Project"

## 1. Check CUDA / the GPU

In [2]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
Sun Oct 11 06:55:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8    

## 2. Download the input from your repo

`passwords.txt` is the program input; `passwords_plain.txt` holds the known
originals and is used only to check the result.

In [3]:
!wget -q $RAW_BASE/Task3/passwords.txt
!wget -q $RAW_BASE/Task3/passwords_plain.txt
!wc -l passwords.txt passwords_plain.txt

 1000 passwords.txt
 1000 passwords_plain.txt
 2000 total


### (Optional) Regenerate test data instead of downloading

`passgen` writes `passwords.txt` and `passwords_plain.txt`. Run this only if
you want fresh data rather than the files from the repo.

In [4]:
%%writefile passgen.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// Task 3 helper - test-data generator (from the lecturer's PasswordGeneratorToText2.c).
// Usage: ./passgen <count>
// Writes passwords.txt (encrypted, crack.cu input) and passwords_plain.txt (originals,
// used only to check results). cudaCrypt is the lecturer's, unchanged.

char* cudaCrypt(char* rawPassword) {

    static char newPassword[11];

    newPassword[0] = rawPassword[0] + 2;
    newPassword[1] = rawPassword[0] - 2;
    newPassword[2] = rawPassword[0] + 1;
    newPassword[3] = rawPassword[1] + 3;
    newPassword[4] = rawPassword[1] - 3;
    newPassword[5] = rawPassword[1] - 1;
    newPassword[6] = rawPassword[2] + 2;
    newPassword[7] = rawPassword[2] - 2;
    newPassword[8] = rawPassword[3] + 4;
    newPassword[9] = rawPassword[3] - 4;
    newPassword[10] = '\0';

    for (int i = 0; i < 10; i++) {
        if (i >= 0 && i < 6) {
            if (newPassword[i] > 122) {
                newPassword[i] = (newPassword[i] - 122) + 97;
            } else if (newPassword[i] < 97) {
                newPassword[i] = (97 - newPassword[i]) + 97;
            }
        } else {
            if (newPassword[i] > 57) {
                newPassword[i] = (newPassword[i] - 57) + 48;
            } else if (newPassword[i] < 48) {
                newPassword[i] = (48 - newPassword[i]) + 48;
            }
        }
    }
    return newPassword;
}

int main(int argc, char **argv) {
    if (argc != 2) {
        fprintf(stderr, "Usage: %s <count>\n", argv[0]);
        return 1;
    }
    int n = atoi(argv[1]);
    if (n <= 0) {
        fprintf(stderr, "Count must be a positive integer\n");
        return 1;
    }

    FILE *fp = fopen("passwords.txt", "w");
    if (fp == NULL) {
        fprintf(stderr, "Could not open passwords.txt\n");
        return 1;
    }
    FILE *fpp = fopen("passwords_plain.txt", "w");
    if (fpp == NULL) {
        fprintf(stderr, "Could not open passwords_plain.txt\n");
        fclose(fp);
        return 1;
    }

    srand((unsigned int) time(NULL));

    char rawPassword[5];
    rawPassword[4] = '\0';

    for (int i = 0; i < n; i++) {
        rawPassword[0] = 'a' + (rand() % 26);
        rawPassword[1] = 'a' + (rand() % 26);
        rawPassword[2] = '0' + (rand() % 10);
        rawPassword[3] = '0' + (rand() % 10);

        char *encrypted = cudaCrypt(rawPassword);
        fprintf(fp,  "%s\n", encrypted);
        fprintf(fpp, "%s\n", rawPassword);
    }

    fclose(fp);
    fclose(fpp);
    printf("Generated %d passwords (passwords.txt + passwords_plain.txt)\n", n);
    return 0;
}


Writing passgen.c


In [5]:
# !gcc -O2 -o passgen passgen.c && ./passgen 10000

## 3. Write the CUDA source

In [6]:
%%writefile crack.cu
#include <stdio.h>
#include <stdlib.h>

// Task 3 - password cracking with CUDA.
// Usage: ./crack <encrypted_file>   Output: decrypted.txt
// Each input line is a 10-char password from the lecturer's cudaCrypt (2 lowercase
// letters + 2 digits = 26*26*10*10 = 67600 candidates). One GPU thread per line
// brute-forces every candidate and writes back the 4-char original on a match.

#define RAW_LEN  4
#define DEC_LEN  5      // raw + '\0'
#define ENC_LEN 10
#define ENC_BUF 11      // encrypted + '\0'

#define CUDA_CHECK(call)                                                     \
    do {                                                                    \
        cudaError_t err = (call);                                          \
        if (err != cudaSuccess) {                                          \
            fprintf(stderr, "CUDA error at %s:%d - %s\n",                  \
                    __FILE__, __LINE__, cudaGetErrorString(err));         \
            exit(1);                                                       \
        }                                                                 \
    } while (0)

// Device version of the lecturer's cudaCrypt; writes into a caller buffer so
// each thread has its own (the original used a shared static buffer).
__device__ void cudaCrypt(const char *rawPassword, char *newPassword) {
    newPassword[0] = rawPassword[0] + 2;
    newPassword[1] = rawPassword[0] - 2;
    newPassword[2] = rawPassword[0] + 1;
    newPassword[3] = rawPassword[1] + 3;
    newPassword[4] = rawPassword[1] - 3;
    newPassword[5] = rawPassword[1] - 1;
    newPassword[6] = rawPassword[2] + 2;
    newPassword[7] = rawPassword[2] - 2;
    newPassword[8] = rawPassword[3] + 4;
    newPassword[9] = rawPassword[3] - 4;
    newPassword[10] = '\0';

    for (int i = 0; i < 10; i++) {
        if (i >= 0 && i < 6) {
            if (newPassword[i] > 122) {
                newPassword[i] = (newPassword[i] - 122) + 97;
            } else if (newPassword[i] < 97) {
                newPassword[i] = (97 - newPassword[i]) + 97;
            }
        } else {
            if (newPassword[i] > 57) {
                newPassword[i] = (newPassword[i] - 57) + 48;
            } else if (newPassword[i] < 48) {
                newPassword[i] = (48 - newPassword[i]) + 48;
            }
        }
    }
}

__global__ void crackKernel(const char *d_enc, char *d_dec, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n) return;

    const char *target = &d_enc[idx * ENC_BUF];

    char candidate[RAW_LEN];
    char attempt[ENC_BUF];

    for (char a = 'a'; a <= 'z'; a++) {
        for (char b = 'a'; b <= 'z'; b++) {
            for (char c = '0'; c <= '9'; c++) {
                for (char d = '0'; d <= '9'; d++) {
                    candidate[0] = a;
                    candidate[1] = b;
                    candidate[2] = c;
                    candidate[3] = d;

                    cudaCrypt(candidate, attempt);

                    int match = 1;
                    for (int k = 0; k < ENC_LEN; k++) {
                        if (attempt[k] != target[k]) {
                            match = 0;
                            break;
                        }
                    }
                    if (match) {
                        d_dec[idx * DEC_LEN + 0] = a;
                        d_dec[idx * DEC_LEN + 1] = b;
                        d_dec[idx * DEC_LEN + 2] = c;
                        d_dec[idx * DEC_LEN + 3] = d;
                        d_dec[idx * DEC_LEN + 4] = '\0';
                        return;
                    }
                }
            }
        }
    }
}

int main(int argc, char **argv) {
    if (argc != 2) {
        fprintf(stderr, "Usage: %s <encrypted_file>\n", argv[0]);
        return 1;
    }
    char *filename = argv[1];

    // Pass 1: count the lines.
    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        fprintf(stderr, "Could not open file: %s\n", filename);
        return 1;
    }
    int n = 0;
    int ch;
    for (ch = getc(fp); ch != EOF; ch = getc(fp)) {
        if (ch == '\n') n++;
    }
    fclose(fp);
    if (n == 0) {
        fprintf(stderr, "File has no passwords\n");
        return 1;
    }

    // Pass 2: read the encrypted passwords into a host buffer.
    char *h_enc = (char *)malloc((size_t)n * ENC_BUF);
    char *h_dec = (char *)malloc((size_t)n * DEC_LEN);
    if (h_enc == NULL || h_dec == NULL) {
        fprintf(stderr, "Host allocation failed\n");
        free(h_enc); free(h_dec);
        return 1;
    }
    fp = fopen(filename, "r");
    if (fp == NULL) {
        fprintf(stderr, "Could not re-open file: %s\n", filename);
        free(h_enc); free(h_dec);
        return 1;
    }
    for (int i = 0; i < n; i++) {
        if (fscanf(fp, "%10s", &h_enc[i * ENC_BUF]) != 1) {
            fprintf(stderr, "Unexpected end of file at line %d\n", i);
            fclose(fp);
            free(h_enc); free(h_dec);
            return 1;
        }
    }
    fclose(fp);

    // GPU memory sized to the file.
    char *d_enc = NULL;
    char *d_dec = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_enc, (size_t)n * ENC_BUF));
    CUDA_CHECK(cudaMalloc((void **)&d_dec, (size_t)n * DEC_LEN));
    CUDA_CHECK(cudaMemcpy(d_enc, h_enc, (size_t)n * ENC_BUF,
                          cudaMemcpyHostToDevice));

    // Split blocks/threads by password count so each thread cracks one password.
    // A fixed 256 threads per block keeps the work spread over multiple blocks
    // (e.g. 1000 passwords -> 4 blocks x 256 threads), as the brief requires.
    int threads = (n < 256) ? n : 256;
    int blocks = (n + threads - 1) / threads;
    printf("Cracking %d passwords with %d blocks x %d threads\n",
           n, blocks, threads);

    crackKernel<<<blocks, threads>>>(d_enc, d_dec, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_dec, d_dec, (size_t)n * DEC_LEN,
                          cudaMemcpyDeviceToHost));

    FILE *out = fopen("decrypted.txt", "w");
    if (out == NULL) {
        fprintf(stderr, "Could not open decrypted.txt for writing\n");
        cudaFree(d_enc); cudaFree(d_dec);
        free(h_enc); free(h_dec);
        return 1;
    }
    for (int i = 0; i < n; i++) {
        fprintf(out, "%s -> %s\n", &h_enc[i * ENC_BUF], &h_dec[i * DEC_LEN]);
    }
    fclose(out);
    printf("Wrote decrypted.txt\n");

    CUDA_CHECK(cudaFree(d_enc));
    CUDA_CHECK(cudaFree(d_dec));
    free(h_enc);
    free(h_dec);
    return 0;
}


Writing crack.cu


## 4. Compile

In [7]:
!nvcc -o crack crack.cu

## 5. Run

Usage: `./crack <encoded_file>`. Output goes to `decrypted.txt`.

In [8]:
!./crack passwords.txt

Cracking 1000 passwords with 1 blocks x 1000 threads
Wrote decrypted.txt


## 6. Inspect the output

In [9]:
!head -10 decrypted.txt

gcflfh7323 -> ei57
bwzrln4012 -> yo26
oknpjl7362 -> mm52
gcffbb5191 -> ec35
wsvxrt5144 -> uu30
jficvx9512 -> hy76
vrukeg8412 -> th66
milcvx1612 -> ky86
njmpjl7312 -> lm56
qmphbd2734 -> oe98


## 7. Check against the known originals

In [10]:
!awk '{print $3}' decrypted.txt > got.txt
!diff got.txt passwords_plain.txt && echo 'ALL MATCH' || echo 'differences above'

ALL MATCH
